# Consulta de reseñas por mapa

En este notebook, el objetivo es realizar una consulta como la de reseñas normal, pero yendo un paso más allá: hacer un *join* con otro DataFrame de Spark de los alojamientos, con las coordenadas de estos, pudiendo graficar los sentimientos de las reseñas (buenas o malas) en un mapa por las coordenadas.

## Importación de librerías

In [1]:
import sys
import pathlib

notebook_dir = pathlib.Path.cwd()
project_root = notebook_dir.parent.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Directorio raíz añadido al path: {project_root}")

Directorio raíz añadido al path: /home/dani/Universidad/3o/2o-cuatri/SDPDII/Practicas/proy_SSDD_II


In [2]:
from src.kafka.consumer_kafka import create_kafka_stream_df, consumer_kafka_avro

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, window
from pyspark.sql.types import StringType
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

import plotly.express as px
import time
from IPython.display import clear_output

In [3]:
# --------- NLP MODEL DEFINITION ---------
@pandas_udf(StringType())
def classify_review(comentarios: pd.Series) -> pd.Series:
    nltk.download('vader_lexicon', quiet=True)
    
    # Initialize VADER analyzer
    sia = SentimentIntensityAnalyzer()
    
    def evaluate_feeling(texto):
        if not texto or pd.isna(texto):
            return "informativa"
        
        # Get VADER polarity scores
        scores = sia.polarity_scores(str(texto))
        
        # Extract compound score
        compound = scores['compound']
        
        # Apply thresholds
        if compound > 0.15: 
            return "buena"
        elif compound < -0.15: 
            return "mala"
        else: 
            return "informativa"
            
    return comentarios.apply(evaluate_feeling)

## Flujo de Spark Streaming

A continuación, se implementa el flujo de Spark Streaming paso a paso. Empezamos definiendo la sesión de Spark.

In [4]:
spark = SparkSession.builder \
    .appName("stream_stream_join_map") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.spark:spark-avro_2.13:4.1.1") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 22:15:35 WARN Utils: Your hostname, dani, resolves to a loopback address: 127.0.1.1; using 192.168.1.86 instead (on interface wlp62s0)
26/05/06 22:15:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/dani/Universidad/3o/2o-cuatri/SDPDII/Practicas/proy_SSDD_II/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/dani/.ivy2.5.2/cache
The jars for the packages stored in: /home/dani/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.spark#spark-avro_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9e15bf0f-cce8-47cf-8ea1-74d9180f5d41;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-ka

Leemos el stream de la tabla principal (topic `airbnb_listings_gold`).

In [5]:
df_listings_stream = create_kafka_stream_df(spark, "airbnb_listings_gold")

df_listings_procesado = df_listings_stream \
    .select(
        col("id").alias("listing_id_lst"), 
        col("latitude").cast("float"), 
        col("longitude").cast("float")
    )

Leemos el stream de la tabla de reseñas (`airbnb_reviews_gold`).

In [6]:
df_reviews_stream = create_kafka_stream_df(spark, "airbnb_reviews_gold")

df_reviews_procesado = df_reviews_stream \
    .withColumn("tipo_review", classify_review(col("comments"))) \
    .withColumn("event_timestamp", col("date").cast("timestamp")) \
    .withWatermark("event_timestamp", "7 days") \
    .filter(col("tipo_review") != "informativa") \
    .select("listing_id", "tipo_review", "event_timestamp")

A continuación realizamos el join entre las dos tablas.

In [7]:
df_join = df_reviews_procesado.join(
    df_listings_procesado, 
    df_reviews_procesado.listing_id == df_listings_procesado.listing_id_lst, 
    "inner"
).select("listing_id", "latitude", "longitude", "tipo_review", "event_timestamp")

Escribimos el DataFrame final en memoria con un nombre de query para poder usarlo.

In [8]:
query_mapa = df_join.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("mapa_reviews_raw") \
    .start()

print("¡Streaming corriendo en segundo plano! Spark está procesando datos hacia la tabla 'mapa_reviews_raw'.")

26/05/06 22:15:44 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-71308f57-0530-4808-aa07-8a4e93738ff0. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/06 22:15:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/06 22:15:45 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


¡Streaming corriendo en segundo plano! Spark está procesando datos hacia la tabla 'mapa_reviews_raw'.


Ejecutamos la siguiente celda (con un bucle infinito) para que vaya recargando los datos que van llegando. Se puede parar la celda cuando queramos.

In [ ]:
print("Iniciando visualización en tiempo real. Presiona el botón de 'Stop' en el Notebook para detener el bucle.")

try:
    while True:
        # Group the data that has arrived up to this exact moment using SQL
        pdf_mapa = spark.sql("""
            SELECT listing_id, latitude, longitude, tipo_review, COUNT(*) as count 
            FROM mapa_reviews_raw 
            GROUP BY listing_id, latitude, longitude, tipo_review
        """).toPandas()
        
        # If joined data has already arrived, draw the map
        if not pdf_mapa.empty:
            clear_output(wait=True) # Clear the previous map to avoid cluttering the screen
            
            # Colors: Green for good, Red for bad
            color_map = {"buena": "#2ecc71", "mala": "#e74c3c"}
            
            fig = px.scatter_mapbox(
                pdf_mapa, 
                lat="latitude", 
                lon="longitude", 
                color="tipo_review",
                size="count", 
                color_discrete_map=color_map,
                hover_name="listing_id",
                hover_data=["count"],
                zoom=11.5, 
                center={"lat": 36.7213, "lon": -4.4214}, # Malaga
                mapbox_style="carto-positron",
                title="🔴 Mapa en Vivo: Sentimiento de Reseñas por Alojamiento"
            )
            
            # Adjust the margins slightly so it looks better in the notebook
            fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
            fig.show()
            
        else:
            print("Esperando a que el Stream-Stream Join empareje los primeros datos...")
            
        # Refresh the chart every 10 seconds
        time.sleep(10)
        
except KeyboardInterrupt:
    print("Visualización detenida por el usuario.")
    
finally:
    pass

/tmp/ipykernel_181376/4277962808.py:19: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


26/05/06 22:20:24 WARN TaskSetManager: Stage 21 contains a task of very large size (1241 KiB). The maximum recommended task size is 1000 KiB.
ERROR:root:KeyboardInterrupt while sending command.               (0 + 20) / 20]
Traceback (most recent call last):
  File "/home/dani/Universidad/3o/2o-cuatri/SDPDII/Practicas/proy_SSDD_II/.venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dani/Universidad/3o/2o-cuatri/SDPDII/Practicas/proy_SSDD_II/.venv/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


Visualización detenida por el usuario.


26/05/06 22:20:41 ERROR Executor: Exception in task 8.0 in stage 21.0 (TID 336)
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.io.BufferedOutputStream.<init>(BufferedOutputStream.java:94)
	at java.base/java.io.BufferedOutputStream.<init>(BufferedOutputStream.java:119)
	at org.apache.spark.storage.DiskBlockObjectWriter$ManualCloseBufferedOutputStream$1.<init>(DiskBlockObjectWriter.scala:157)
	at org.apache.spark.storage.DiskBlockObjectWriter.initialize(DiskBlockObjectWriter.scala:159)
	at org.apache.spark.storage.DiskBlockObjectWriter.open(DiskBlockObjectWriter.scala:168)
	at org.apache.spark.storage.DiskBlockObjectWriter.write(DiskBlockObjectWriter.scala:333)
	at org.apache.spark.shuffle.sort.BypassMergeSortShuffleWriter.write(BypassMergeSortShuffleWriter.java:185)
	at org.apache.spark.shuffle.ShuffleWriteProcessor.write(ShuffleWriteProcessor.scala:57)
	at org.apache.spark.scheduler.ShuffleMapTask.runTask(ShuffleMapTask.scala:111)
	at org.apache.spark.scheduler.ShuffleM